In [1]:
import pandas as pd

path = "../data/raw/Mobile_Parts_Wholesale_Dataset.xlsx"
po = pd.read_excel(path, sheet_name="Purchase_Orders")
suppliers = pd.read_excel(path, sheet_name="Suppliers")
returns = pd.read_excel(path, sheet_name="Returns_Complaints")
products = pd.read_excel(path, sheet_name="Products")

po['order_date'] = pd.to_datetime(po['order_date'])
po['promised_delivery_date'] = pd.to_datetime(po['promised_delivery_date'])
po['actual_delivery_date'] = pd.to_datetime(po['actual_delivery_date'])
returns['return_date'] = pd.to_datetime(returns['return_date'])

print(po.shape, returns.shape)
returns.head()

(298, 9) (180, 6)


,return_id,order_id,product_id,return_date,reason,resolved
0,R0001,O000320,P0104,2025-05-28,Wrong Part Sent,Yes
1,R0002,O002403,P0079,2025-11-27,Damaged in Transit,Yes
2,R0003,O004473,P0097,2026-04-15,Dead on Arrival,Yes
3,R0004,O002827,P0152,2025-12-30,Dead on Arrival,Yes
4,R0005,O000085,P0053,2025-03-13,Touch Not Working,Yes


In [2]:
# Map each product to its most frequent supplier (a product may have been bought from multiple suppliers)
product_supplier_map = (
    po.groupby('product_id')['supplier_id']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={'supplier_id': 'primary_supplier_id'})
)

returns_with_supplier = returns.merge(product_supplier_map, on='product_id', how='left')
print(returns_with_supplier.shape)
returns_with_supplier.head()

(180, 7)


,return_id,order_id,product_id,return_date,reason,resolved,primary_supplier_id
0,R0001,O000320,P0104,2025-05-28,Wrong Part Sent,Yes,NaN
1,R0002,O002403,P0079,2025-11-27,Damaged in Transit,Yes,S011
2,R0003,O004473,P0097,2026-04-15,Dead on Arrival,Yes,S005
3,R0004,O002827,P0152,2025-12-30,Dead on Arrival,Yes,S004
4,R0005,O000085,P0053,2025-03-13,Touch Not Working,Yes,S013


In [3]:
# Total units purchased per supplier (denominator for a fair rate, not just raw count)
units_purchased = po.groupby('supplier_id')['quantity_ordered'].sum().reset_index()
units_purchased.columns = ['supplier_id', 'total_units_purchased']

# Total returns attributed to each supplier
return_counts = (
    returns_with_supplier.groupby('primary_supplier_id')
    .size()
    .reset_index(name='return_count')
    .rename(columns={'primary_supplier_id': 'supplier_id'})
)

supplier_quality = units_purchased.merge(return_counts, on='supplier_id', how='left')
supplier_quality['return_count'] = supplier_quality['return_count'].fillna(0)
supplier_quality['return_rate_pct'] = (
    supplier_quality['return_count'] / supplier_quality['total_units_purchased'] * 100
).round(2)

supplier_quality.sort_values('return_rate_pct', ascending=False)

,supplier_id,total_units_purchased,return_count,return_rate_pct
3,S004,3079,17,0.55
13,S014,3393,13,0.38
12,S013,2733,10,0.37
4,S005,2833,10,0.35
5,S006,4009,14,0.35
1,S002,4548,15,0.33
8,S009,3807,12,0.32
6,S007,4197,12,0.29
11,S012,4901,13,0.27
7,S008,4524,12,0.27


In [4]:
delivered = po[po['status'] != 'Pending'].copy()
delivered['on_time'] = delivered['actual_delivery_date'] <= delivered['promised_delivery_date']

base_scorecard = delivered.groupby('supplier_id').agg(
    total_orders=('po_id', 'count'),
    on_time_pct=('on_time', 'mean'),
    avg_unit_cost=('unit_cost_inr', 'mean')
).reset_index()
base_scorecard['on_time_pct'] = (base_scorecard['on_time_pct'] * 100).round(1)

scorecard = base_scorecard.merge(
    supplier_quality[['supplier_id', 'return_rate_pct']], on='supplier_id', how='left'
)
scorecard = scorecard.merge(suppliers[['supplier_id', 'supplier_name', 'country']], on='supplier_id')

scorecard.sort_values('return_rate_pct', ascending=True)

,supplier_id,total_orders,on_time_pct,avg_unit_cost,return_rate_pct,supplier_name,country
0,S001,22,63.6,524.697273,0.13,Toor Inc Electronics,China
2,S003,19,68.4,475.937895,0.14,Oommen Ltd Trading Co.,China
10,S011,22,63.6,584.844545,0.22,Karpe and Sons Trading Co.,India
9,S010,25,60.0,317.595600,0.26,"Peri, Grewal and Bhavsar Trading Co.",India
7,S008,21,66.7,1028.004286,0.27,Sachar-Sen Electronics,China
11,S012,29,69.0,438.362414,0.27,"Das, Misra and Bava Electronics",China
6,S007,21,71.4,354.422857,0.29,Goda-Anand Trading Co.,China
8,S009,19,78.9,518.955263,0.32,Sathe Ltd Trading Co.,China
1,S002,25,76.0,707.209600,0.33,"Dass, Magar and Dugar Trading Co.",India
5,S006,23,73.9,412.950870,0.35,Sekhon-Radhakrishnan Electronics,India


In [5]:
# Normalize each metric to a 0-100 scale, higher = better, before weighting
scorecard['on_time_score'] = scorecard['on_time_pct']  # already 0-100, higher is better
scorecard['cost_score'] = 100 - (
    (scorecard['avg_unit_cost'] - scorecard['avg_unit_cost'].min()) /
    (scorecard['avg_unit_cost'].max() - scorecard['avg_unit_cost'].min()) * 100
)  # lower cost = higher score
scorecard['quality_score'] = 100 - (
    (scorecard['return_rate_pct'] - scorecard['return_rate_pct'].min()) /
    (scorecard['return_rate_pct'].max() - scorecard['return_rate_pct'].min() + 1e-9) * 100
)  # lower return rate = higher score

# Weights: on-time delivery matters most, then cost, then return/quality signal
scorecard['composite_score'] = (
    scorecard['on_time_score'] * 0.5 +
    scorecard['cost_score'] * 0.3 +
    scorecard['quality_score'] * 0.2
).round(1)

scorecard_final = scorecard[['supplier_id', 'supplier_name', 'country', 'total_orders',
                              'on_time_pct', 'avg_unit_cost', 'return_rate_pct', 'composite_score']]
scorecard_final = scorecard_final.sort_values('composite_score', ascending=False)
scorecard_final

,supplier_id,supplier_name,country,total_orders,on_time_pct,avg_unit_cost,return_rate_pct,composite_score
2,S003,Oommen Ltd Trading Co.,China,19,68.4,475.937895,0.14,77.0
6,S007,Goda-Anand Trading Co.,China,21,71.4,354.422857,0.29,76.5
9,S010,"Peri, Grewal and Bhavsar Trading Co.",India,25,60.0,317.595600,0.26,73.8
4,S005,"Gala, Agate and Ravi Trading Co.",India,16,75.0,406.018125,0.35,73.3
0,S001,Toor Inc Electronics,China,22,63.6,524.697273,0.13,73.1
11,S012,"Das, Misra and Bava Electronics",China,29,69.0,438.362414,0.27,72.7
5,S006,Sekhon-Radhakrishnan Electronics,India,23,73.9,412.950870,0.35,72.4
8,S009,Sathe Ltd Trading Co.,China,19,78.9,518.955263,0.32,71.9
12,S013,Sarma LLC Electronics,India,16,75.0,523.118750,0.37,67.4
10,S011,Karpe and Sons Trading Co.,India,22,63.6,584.844545,0.22,66.2


In [6]:
scorecard_final.to_csv("../reports/supplier_scorecard.csv", index=False)
print("Saved updated supplier_scorecard.csv with return-rate and composite score")

Saved updated supplier_scorecard.csv with return-rate and composite score
